# model matrix objective promotion analysis overview v1

目标：验证 `src/objective` 和 `src/analysis` 已接入 model matrix、analysis manifest、dashboard export 和 schedule 摘要。重点检查 objective/promotion 分数，以及 regime/window 模型差异 overview 是否进入 dashboard/schedule 链路。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dashboard import write_model_matrix_dashboard_export
from src.model.matrix import run_factor_feature_set_matrix
from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

EXPERIMENT = "model_matrix_objective_promotion_v1"
CARD = ROOT / "research" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

In [2]:
outputs = run_factor_feature_set_matrix(
    card_dir=CARD,
    start="2026-01-01",
    end="2026-06-09",
    train_end="2026-04-30",
    model_names=("ridge_mse", "random_forest_clf"),
    top_n_factors=3,
)
dashboard_payload = write_model_matrix_dashboard_export(
    CARD,
    output_path=CARD / "dashboard_export.json",
    top_n=5,
)

summary = json.loads((CARD / "analysis_summary.json").read_text(encoding="utf-8"))
manifest = json.loads((CARD / "analysis_manifest.json").read_text(encoding="utf-8"))
objective_scores = pd.read_csv(CARD / "objective_scores.csv")
promotion_scores = pd.read_csv(CARD / "promotion_scores.csv")

summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'adapter_status': 'paper_strict_adapter',
 'adapter_statuses': ['paper_strict_adapter'],
 'analysis_layers': ['regime_metrics',
  'window_metrics',
  'window_regime_metrics',
  'qlib_risk_metrics',
  'model_effect_deltas',
  'objective_scores',
  'promotion_scores',
  'model_comparison',
  'qlib_return_frame',
  'segment_metrics',
  'segment_effect_matrix',
  'model_review_briefing',
  'failure_modes'],
 'analysis_overview': {'baseline_model': 'ridge_mse__ic_weighted_composite',
  'dashboard_note': '统一 regime/window 模型差异摘要，供 dashboard 总览和 schedule 通知读取。',
  'decision_hint_counts': {'block_or_rework': 24,
   'cost_review': 2,
   'promote_candidate': 6,
   'reduce_or_guardrail': 2},
  'largest_relative_return_spread_segments': [{'baseline_model': 'ridge_mse__ic_weighted_composite',
    'best_relative_return_model': 'random_forest_clf__ic_weighted_composite',
    'best_relative_return_vs_baseline_delta': 3.8238407145837456,
    'best_stability_model': 'random_forest_clf__all_features',
 

In [3]:
events: list[str] = []

analysis_overview = dashboard_payload["analysis_overview"]
top_gap = (analysis_overview.get("largest_relative_return_spread_segments") or [{}])[0]
top_weakness = analysis_overview.get("top_weakness") or {}


def matrix_runner(**kwargs):
    return {"variant_count": int(summary["variant_count"]), "status": summary["status"]}


def dashboard_exporter(card_dir: Path, *, output_path: Path, top_n: int):
    output_path.write_text(json.dumps(dashboard_payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return dashboard_payload


config = ModelDashboardPipelineConfig(
    name="model_dashboard_refresh",
    card_dir=CARD,
    dashboard_output_path=CARD / "scheduled_dashboard_export.json",
    start="2026-01-01",
    end="2026-06-09",
    train_end="2026-04-30",
    model_names=("ridge_mse", "random_forest_clf"),
    notify_details=True,
)
result = run_model_dashboard_refresh_once(
    config,
    lock_dir=CARD / "locks",
    notify=events.append,
    matrix_runner=matrix_runner,
    dashboard_exporter=dashboard_exporter,
)

notebook_summary = {
    "schema_version": "research.model_matrix_objective_promotion_v1",
    "status": "completed",
    "matrix_status": summary["status"],
    "variant_count": int(summary["variant_count"]),
    "analysis_overview_schema": analysis_overview.get("schema_version", ""),
    "analysis_top_gap_segment": top_gap.get("segment", ""),
    "analysis_top_gap_best_model": top_gap.get("best_relative_return_model", ""),
    "analysis_top_gap_worst_model": top_gap.get("worst_relative_return_model", ""),
    "analysis_top_weakness_model": top_weakness.get("model", ""),
    "analysis_top_weakness_segment": top_weakness.get("segment", ""),
    "objective_model_count": int(summary["objective_evaluation"]["model_count"]),
    "promotion_model_count": int(summary["promotion_summary"]["model_count"]),
    "objective_score_rows": int(len(objective_scores)),
    "promotion_score_rows": int(len(promotion_scores)),
    "manifest_paths": {
        "objective_scores": manifest["paths"].get("objective_scores", ""),
        "promotion_scores": manifest["paths"].get("promotion_scores", ""),
    },
    "dashboard_objective_rows": len(dashboard_payload["objective"]["scores"]),
    "dashboard_promotion_rows": len(dashboard_payload["objective"]["promotion_scores"]),
    "notification": events[0] if events else "",
    "notification_contains_regime_gap": "regime_gap=" in (events[0] if events else ""),
    "notification_contains_promotion": "promotion=" in (events[0] if events else ""),
}
(CARD / "notebook_summary.json").write_text(json.dumps(notebook_summary, ensure_ascii=False, indent=2), encoding="utf-8")
(CARD / "notification_events.json").write_text(json.dumps(events, ensure_ascii=False, indent=2), encoding="utf-8")
notebook_summary

{'schema_version': 'research.model_matrix_objective_promotion_v1',
 'status': 'completed',
 'matrix_status': 'feature_set_matrix_completed',
 'variant_count': 6,
 'analysis_overview_schema': 'analysis.overview.v1',
 'analysis_top_gap_segment': 'downtrend',
 'analysis_top_gap_best_model': 'random_forest_clf__ic_weighted_composite',
 'analysis_top_gap_worst_model': 'ridge_mse__all_features',
 'analysis_top_weakness_model': 'ridge_mse__all_features',
 'analysis_top_weakness_segment': 'stress',
 'objective_model_count': 6,
 'promotion_model_count': 6,
 'objective_score_rows': 6,
 'promotion_score_rows': 6,
 'manifest_paths': {'objective_scores': 'objective_scores.csv',
  'promotion_scores': 'promotion_scores.csv'},
 'dashboard_objective_rows': 6,
 'dashboard_promotion_rows': 6,
 'notification': 'model_dashboard_refresh:success | top_model=ridge_mse__ic_weighted_composite | dashboard_manifest=analysis_manifest | attempts=1 | export=/Users/rziiiii/dev/workspace/research/model_matrix_objectiv

In [4]:
assert result.status == "success"
assert notebook_summary["analysis_overview_schema"] == "analysis.overview.v1"
assert notebook_summary["analysis_top_gap_segment"]
assert notebook_summary["analysis_top_gap_best_model"]
assert notebook_summary["analysis_top_gap_worst_model"]
assert notebook_summary["analysis_top_weakness_model"]
assert notebook_summary["objective_model_count"] == notebook_summary["variant_count"]
assert notebook_summary["promotion_model_count"] == notebook_summary["variant_count"]
assert notebook_summary["objective_score_rows"] == notebook_summary["variant_count"]
assert notebook_summary["promotion_score_rows"] == notebook_summary["variant_count"]
assert notebook_summary["manifest_paths"]["objective_scores"] == "objective_scores.csv"
assert notebook_summary["manifest_paths"]["promotion_scores"] == "promotion_scores.csv"
assert notebook_summary["dashboard_objective_rows"] == notebook_summary["variant_count"]
assert notebook_summary["dashboard_promotion_rows"] == notebook_summary["variant_count"]
assert notebook_summary["notification_contains_regime_gap"] is True
assert notebook_summary["notification_contains_promotion"] is True

## 中文结论

- 做了什么：验证 `src/objective` 的 objective score/promotion score，以及 `src/analysis` 的 `analysis_overview` 已接入 model matrix、analysis manifest、dashboard export 和 schedule 通知。
- 核心指标：样例包含 2 个基础模型、3 个因子集、6 个模型变体；`objective_scores.csv` 和 `promotion_scores.csv` 均为 6 行，并通过 manifest 公开给 dashboard。
- regime 差异：`analysis_overview.v1` 从 `segment_effect_matrix` 和 `model_review_briefing` 汇总最大 regime/window 模型差异；本次样例的最大 gap 和 top weakness 已写入 `notebook_summary.json`，schedule 通知包含 `regime_gap=`。
- 是否晋升：晋升为 objective/promotion/analysis 框架链路能力；样例中的 promotion decision 不是 production 策略晋升结论。
- 风险：当前 promotion 使用框架默认阈值，analysis overview 只负责暴露分段差异，不替代 production notebook 对成本、压力窗口和阈值的固定判断。
- 下一步：production notebook 固定策略运行时复用同一 objective/backtest/analysis/dashboard manifest 链路，skill 只保留抽象研究思路，不再承载项目功能状态。